# Enrollment API — demo (BT2)

Live calls against the running FastAPI server, for the video demo. It walks through:

- successful reads (**200 OK**, REST maturity level 2)
- registering a course (**201 Created**) with an **Idempotency-Key**
- **idempotency**: a retry with the same key returns the same enrollment
- error handling: **404** and **422** as RFC 7807 `problem+json`
- **API versioning**: the same student under `v1` and `v2`
- cancelling a registration (**204 No Content**)

**Before running** — start the stack and seed the database, then run the cells top to bottom:

```bash
make up && make seed        # API at http://localhost:8000  (Swagger UI at /docs)
```

The notebook only needs `httpx` (already a project dependency). To get a Jupyter
kernel: `uv sync --extra web --extra notebook`.

In [ ]:
# Setup: one HTTP client + a helper that prints status code, content-type and body.
import json
import uuid

import httpx

BASE = "http://localhost:8000"
V1 = f"{BASE}/api/v1"
V2 = f"{BASE}/api/v2"

client = httpx.Client(timeout=10.0)


def show(resp: httpx.Response, note: str = "") -> httpx.Response:
    """Pretty-print a response: method, URL, status, content-type, JSON body."""
    if note:
        print(f"# {note}")
    print(f"{resp.request.method} {resp.request.url}")
    print(f"-> {resp.status_code} {resp.reason_phrase}  [{resp.headers.get('content-type', '')}]")
    if resp.content:
        try:
            print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
        except ValueError:
            print(resp.text)
    print()
    return resp


def as_list(payload):
    """Some list endpoints return a bare list, others a {"items": [...]} page."""
    if isinstance(payload, dict) and "items" in payload:
        return payload["items"]
    return payload


# Quick sanity check that the server is up.
show(client.get(f"{BASE}/health"), "health check")

## 1. Successful reads (200 OK)

Read-only endpoints: list terms (one has its registration window open), fetch a
single student, and browse offerings with server-side pagination.

In [ ]:
show(client.get(f"{V1}/terms"), "list terms")
show(client.get(f"{V1}/students/1"), "get one student")
show(
    client.get(f"{V1}/offerings", params={"open_only": True, "page": 1, "size": 3}),
    "browse offerings (paginated)",
)

## 2. Register a course (201 Created) + Idempotency-Key

We pick, for a chosen student, an offering in the currently-open term that they can
still register for — open, seats available, not already taken, and no schedule
clash — so the `POST` returns a clean **201**. Sending an `Idempotency-Key` makes
the request safe to retry (next section).

In [ ]:
STUDENT_ID = 1

# Find the term whose registration window is currently open.
terms = as_list(client.get(f"{V1}/terms").json())
open_term = next(t for t in terms if t["is_registration_open"])
print("open term:", open_term["id"], "-", open_term["name"])

# The student's current registrations in that term -> what is taken + busy time slots.
current = client.get(
    f"{V1}/students/{STUDENT_ID}/enrollments",
    params={"status": "REGISTERED", "term_id": open_term["id"]},
).json()
taken = {e["offering_id"] for e in current}
busy = {
    (e["offering"]["day_of_week"], e["offering"]["start_time"], e["offering"]["end_time"])
    for e in current
}


def clashes(offering) -> bool:
    """True if the offering overlaps any already-registered class this term."""
    day, start, end = offering["day_of_week"], offering["start_time"], offering["end_time"]
    return any(bd == day and not (end <= bs or start >= be) for bd, bs, be in busy)


# First registrable, non-conflicting offering in the open term.
offerings = as_list(
    client.get(f"{V1}/offerings", params={"term_id": open_term["id"], "page": 1, "size": 100}).json()
)
offering = next(
    o for o in offerings if o["can_register"] and o["id"] not in taken and not clashes(o)
)
print("chosen offering:", offering["id"], "-", offering["course"]["course_code"])

# Register -> 201 Created. Keep the key so we can replay it in the next cell.
idem_key = str(uuid.uuid4())
resp = show(
    client.post(
        f"{V1}/enrollments",
        json={"student_id": STUDENT_ID, "offering_id": offering["id"]},
        headers={"Idempotency-Key": idem_key},
    ),
    "POST /enrollments with Idempotency-Key",
)
enrollment = resp.json()

## 3. Idempotency: retry with the same key

The same request with the **same** `Idempotency-Key` replays the original result
instead of creating a duplicate — the returned `id` is identical.

In [ ]:
retry = show(
    client.post(
        f"{V1}/enrollments",
        json={"student_id": STUDENT_ID, "offering_id": offering["id"]},
        headers={"Idempotency-Key": idem_key},
    ),
    "same key -> replay, not a new row",
)
print("same id?", enrollment["id"], "==", retry.json()["id"], "->", enrollment["id"] == retry.json()["id"])

## 4. Error handling — 404 (RFC 7807)

Requesting a resource that does not exist returns **404** as
`application/problem+json`.

In [ ]:
show(client.get(f"{V1}/students/999999"), "GET a student that does not exist")

## 5. Error handling — 422 (RFC 7807)

A request body that fails validation (here `offering_id` is missing) returns
**422**, also `problem+json`, with an `errors` array pointing at the bad field.

In [ ]:
show(
    client.post(f"{V1}/enrollments", json={"student_id": STUDENT_ID}),
    "POST with a missing field",
)

## 6. API versioning — v1 vs v2

The same student under two contracts mounted side by side: `v1` exposes
`student_code` / `full_name` / timestamps; `v2` renames them to `code` / `name`
and drops the timestamps.

In [ ]:
show(client.get(f"{V1}/students/1"), "v1 contract")
show(client.get(f"{V2}/students/1"), "v2 contract")

## 7. Cleanup — cancel the registration (204)

Drop the demo registration so the notebook stays re-runnable. `DELETE` returns
**204 No Content**.

In [ ]:
show(client.delete(f"{V1}/enrollments/{enrollment['id']}"), "cancel (drop) the registration")
client.close()